**Notebook for Creating Evaluation Baseline**

In [ ]:
import json
import time
from langchain_openai import ChatOpenAI


DEEPSEEK_MODEL = "deepseek-chat"
DEEPSEEK_KEY = ""

In [ ]:
FACET_JSON = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions/final_output/141026-cs/2303.13743.json"
# use any json from google drive as shared in FACET-Data
with open(FACET_JSON) as f:
    facet_data = json.load(f)

In [3]:
facet_data.keys()

dict_keys(['m_id', 'm_title', 'm_abstract', 'm_aspects', 'references', 'facet_ranking', 'facet_scoring'])

In [ ]:
# this function generates a M R pair as shown below
def get_m_r_pair(facet_data):
    m_title = facet_data['m_title']
    m_abstract = facet_data['m_abstract']
    m_aspects = facet_data['m_aspects']
    for r in facet_data['references']: # r is the ref paper key
        ref_data = facet_data['references'][r][0]
        r_abstract = ref_data[r]['abstract']
        r_title = ref_data[r]['title']
        r_aspects = ref_data[r]['aspect']
        yield (m_title,m_abstract,m_aspects),(r_title,r_abstract,r_aspects)

In [28]:
nb_refs = len(facet_data['references'])
print(f"Total {nb_refs} refernces")
mr_data = get_m_r_pair(facet_data)
for i in range(nb_refs):
    m_data,r_data = next(mr_data)
    print(r_data[0]) # prints title of ref

Total 22 refernces
Universal Correspondence Network
ShapeNet: An Information-Rich 3D Model Repository
Optimizing the Latent Space of Generative Networks
StyleNeRF: A Style-based 3D-Aware Generator for High-resolution Image
  Synthesis
NerfDiff: Single-image View Synthesis with NeRF-guided Distillation from
  3D-aware Diffusion
nerf2nerf: Pairwise Registration of Neural Radiance Fields
Progressive Growing of GANs for Improved Quality, Stability, and
  Variation
GET3D: A Generative Model of High Quality 3D Textured Shapes Learned
  from Images
Adam: A Method for Stochastic Optimization
3D-aware Blending with Generative NeRFs
Joint-task Self-supervised Learning for Temporal Correspondence
Learning Implicit Functions for Topology-Varying Dense 3D Shape
  Correspondence
Alias-Free Generative Adversarial Networks
Point-E: A System for Generating 3D Point Clouds from Complex Prompts
Implicit Neural Representations with Periodic Activation Functions
3D GAN Inversion for Controllable Portrait I

In [ ]:
def ask_deepseek(prompt, system_prompt="You are a helpful assistant.",
                  model=DEEPSEEK_MODEL, temperature=0.0, max_retries=3):
    """Call DeepSeek (OpenAI-compatible endpoint) with any ad-hoc prompt;
    returns the plain text response. Standalone -- no dependency on any
    other script in this repo."""
    llm = ChatOpenAI(
        model=model,
        temperature=temperature,
        api_key=DEEPSEEK_KEY,
        base_url="https://api.deepseek.com",
    )
    messages = [("system", system_prompt), ("human", prompt)]
    last_err = None
    for attempt in range(max_retries):
        try:
            return llm.invoke(messages).content
        except Exception as ex:
            last_err = ex
            wait = 2 ** attempt
            print(f"  LLM error (attempt {attempt+1}/{max_retries}): {ex} -> retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"DeepSeek call failed after {max_retries} attempts: {last_err}")


# Example:
# response = ask_deepseek("Summarize in one sentence: " + m_data[1])
# print(response)